# 🔥 FireWatch — Train on a free Colab GPU

Trains the **TensorFlow / KerasHub RetinaNet** fire+smoke detector on Google's free GPU,
then saves the weights to your Google Drive so you can detect locally.

### Before you start (one-time)
1. **Runtime → Change runtime type → Hardware accelerator = GPU → Save.**
2. On your PC, zip the prepared dataset folder `data/dfire` into **`dfire.zip`**
   (a ready-made `dfire.zip` is at your project root).
3. Upload **`dfire.zip`** to the top level of your Google Drive ("My Drive").

Then just run the cells top to bottom (**Runtime → Run all**).

## 1. Confirm the GPU is on

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import tensorflow as tf
print('TensorFlow', tf.__version__, '| GPUs:', tf.config.list_physical_devices('GPU'))
assert tf.config.list_physical_devices('GPU'), 'No GPU! Runtime > Change runtime type > GPU, then rerun.'

## 2. Get the code + install KerasHub
Clones your GitHub repo (the `master` branch, which has all the code) and installs the
detection deps. If pip prints a "RESTART RUNTIME" button, click it, then run this cell again.

In [ ]:
import os
if not os.path.isdir('fire-detection'):
    !git clone --depth 1 -b master https://github.com/01End/fire-detection.git
%cd fire-detection
!pip install -q -U "tensorflow>=2.19" keras-hub opencv-python-headless
import keras_hub, keras
print('keras', keras.__version__, '| keras_hub', keras_hub.__version__)

## 3. Mount Google Drive and unzip the dataset
A popup will ask you to authorize access to your Drive — click through it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ZIP = '/content/drive/MyDrive/dfire.zip'   # change if you put it elsewhere
assert os.path.exists(ZIP), f'Not found: {ZIP}  (upload dfire.zip to My Drive)'
!mkdir -p data && unzip -q -o "$ZIP" -d data/
# Expect: data/dfire/train/images, data/dfire/val/images, data/dfire/class_map.json
!ls data/dfire && echo '--- train imgs:' && ls data/dfire/train/images | wc -l && echo '--- val imgs:' && ls data/dfire/val/images | wc -l

## 4. Train on the GPU 🚀
~20–30 min for the 3000/600 set on a T4. Edit `--epochs` / `--image-size` to taste.
RetinaNet starts from COCO‑pretrained weights, so it learns fire/smoke quickly.

In [ ]:
!python -m training.tf_train \
    --data data/dfire \
    --epochs 12 \
    --batch-size 8 \
    --image-size 512 \
    --lr 0.0005 \
    --out models/fire_retinanet.weights.h5

## 5. Save the trained weights to your Drive (so you can download them)

In [ ]:
!cp models/fire_retinanet.weights.h5 /content/drive/MyDrive/fire_retinanet.weights.h5
print('✅ Saved to My Drive → fire_retinanet.weights.h5  (download it from drive.google.com)')

## 6. See it work — detect on validation images 🔥
Draws boxes on a few val images and shows them right here in the notebook.

In [ ]:
import glob, shutil, os
os.makedirs('val_sample', exist_ok=True)
for p in sorted(glob.glob('data/dfire/val/images/*'))[:8]:
    shutil.copy(p, 'val_sample/')

!python -m firewatch.cli detect \
    --model models/fire_retinanet.weights.h5 \
    --input val_sample \
    --backend tf --arch retinanet \
    --score-thr 0.3 \
    --out detect_out

from IPython.display import Image, display
for p in sorted(glob.glob('detect_out/*'))[:8]:
    print(p); display(Image(p))